In [1]:
from transformers import AutoTokenizer, GPT2LMHeadModel, GPT2Config

In [10]:
tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token

In [3]:
config = GPT2Config(vocab_size=len(tokenizer))
model = GPT2LMHeadModel(config)

In [4]:
with open("dataset.txt", "r") as file:
    data = file.read()

In [5]:
from datasets import load_dataset

dataset = load_dataset("text", data_files={"train": "dataset.txt"})

In [6]:
def tokenize(example):
    return tokenizer(example["text"])

tokenized_ds = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

In [7]:
block_size = 1024

def group_texts(examples):
    # Concatenate all the token arrays in the current batch
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # Drop the small remainder of tokens at the very end of the dataset
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
        
    # Split the massive concatenated array into blocks of 'block_size'
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    
    # For training from scratch (causal modeling), the target labels are the exact same as the inputs.
    # The model handles shifting the tokens by one position internally during the forward pass.
    result["labels"] = result["input_ids"].copy()
    return result

lm_datasets = tokenized_ds.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

print(lm_datasets["train"][0].keys()) 

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [ ]:
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer, 
    mlm=False
)

training_args = TrainingArguments(
    output_dir="./gpt2-trained-from-scratch",
    logging_steps=50,
    learning_rate=5e-4,
    weight_decay=0.1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    save_strategy="steps",
    save_steps=1000,
    fp16=True,
)

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=lm_datasets["train"],
    data_collator=data_collator,
)

trainer.train()

In [ ]:
def generate_text(query, model, tokenizer, max_new_tokens=50, device=None):
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    model.eval()
    model.to(device)
    
    inputs = tokenizer(query, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,      # How many new tokens to generate
            do_sample=True,                     # Enable sampling (adds randomness/creativity)
            temperature=0.8,                    # Higher = more random; Lower = more predictable
            top_k=50,                           # Limits sampling to the top 50 most likely tokens
            pad_token_id=tokenizer.eos_token_id # Prevents warnings since GPT-2 lacks a native pad token
        )
        
    generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    return generated_text

generate_text("nima", model, tokenizer)